# Transformers NSFW Nano

In [ ]:
import os
from transformers import AutoImageProcessor, AutoModelForImageClassification
from PIL import Image
import torch

# Define local directory to save the model
model_name = "Marqo/nsfw-image-detection-384"
local_model_path = "./nsfw-image-detection-384"

# Download and save model and processor locally
print(f"Downloading model to {local_model_path}...")
processor = AutoImageProcessor.from_pretrained(model_name, use_fast=False)
model = AutoModelForImageClassification.from_pretrained(model_name)

# Save to local directory
processor.save_pretrained(local_model_path)
model.save_pretrained(local_model_path)
print(f"Model saved successfully to {local_model_path}")

# Now load from local directory
print("\nLoading model from local directory...")
processor = AutoImageProcessor.from_pretrained(local_model_path, use_fast=False)
model = AutoModelForImageClassification.from_pretrained(local_model_path)
model.eval()

# Get image files
image_files = [f for f in os.listdir('./test_images') if f.lower().endswith(('.jpg', '.jpeg', '.png'))]

print(f"\n{'Image':<30} | {'Prediction':<10} | {'Confidence':>10} | Details")
print("-" * 90)

for img_file in image_files:
    img = Image.open('./test_images/' + img_file).convert("RGB")
    
    with torch.no_grad():
        inputs = processor(images=img, return_tensors="pt")
        probs = torch.softmax(model(**inputs).logits, dim=-1)[0]
    
    pred_id = int(probs.argmax())
    label = model.config.id2label[pred_id]
    
    # Format all probabilities
    details = " | ".join([f"{model.config.id2label[i]}: {p:.2%}" for i, p in enumerate(probs)])
    
    print(f"{img_file:<30} | {label:<10} | {probs[pred_id]:>9.2%} | {details}")

# HUGGING FACE TO ONNX

In [ ]:
"""
PyTorch to ONNX Conversion Script for NSFW Detection Model
This script converts the Hugging Face PyTorch model to ONNX format with verification.
"""

import os
import torch
import onnx
import onnxruntime as ort
import numpy as np
from transformers import AutoImageProcessor, AutoModelForImageClassification
from PIL import Image


def convert_to_onnx(
    model_path: str = "./nsfw-image-detection-384",
    output_path: str = "./nsfw-image-detection-384.onnx",
    opset_version: int = 18,  # Use opset 18 to avoid conversion issues
    test_image_path: str = "./test_images/normal_1.jpg"
):
    """
    Convert a Hugging Face PyTorch model to ONNX format.
    
    Args:
        model_path: Path to the saved PyTorch model
        output_path: Path where ONNX model will be saved
        opset_version: ONNX opset version (default: 14)
        test_image_path: Path to test image for verification
    """
    
    print("=" * 80)
    print("PyTorch to ONNX Conversion")
    print("=" * 80)
    
    # Step 1: Load the model and processor
    print("\n[1/5] Loading PyTorch model and processor...")
    processor = AutoImageProcessor.from_pretrained(model_path, use_fast=False)
    model = AutoModelForImageClassification.from_pretrained(model_path)
    model.eval()
    print("✓ Model loaded successfully")
    
    # Step 2: Create dummy input for export
    print("\n[2/5] Creating dummy input tensor...")
    # Get expected input size from processor
    input_size = 384  # Default to 224 if not specified
    batch_size = 1
    channels = 3
    
    dummy_input = torch.randn(batch_size, channels, input_size, input_size)
    print(f"✓ Dummy input shape: {dummy_input.shape}")
    
    # Step 3: Export to ONNX
    print("\n[3/5] Exporting model to ONNX format...")
    # Use legacy exporter for better compatibility with transformers models
    with torch.no_grad():
        torch.onnx.export(
            model,
            dummy_input,
            output_path,
            export_params=True,
            opset_version=opset_version,
            do_constant_folding=True,  # Optimize constant operations
            input_names=['pixel_values'],
            output_names=['logits'],
            dynamo=False  # Use legacy exporter for better accuracy
        )
    print(f"✓ Model exported to: {output_path}")
    
    # Step 4: Verify ONNX model
    print("\n[4/5] Verifying ONNX model...")
    onnx_model = onnx.load(output_path)
    onnx.checker.check_model(onnx_model)
    print("✓ ONNX model is valid")
    
    # Step 5: Test ONNX inference and compare with PyTorch
    print("\n[5/5] Testing ONNX inference...")
    ort_session = ort.InferenceSession(output_path)
    
    # Load a test image
    if os.path.exists(test_image_path):
        test_image = Image.open(test_image_path).convert("RGB")
        
        # Prepare input for both models
        inputs = processor(images=test_image, return_tensors="pt")
        pixel_values = inputs['pixel_values'].numpy()
        
        # PyTorch inference
        with torch.no_grad():
            pytorch_output = model(**inputs).logits
            pytorch_probs = torch.softmax(pytorch_output, dim=-1)[0]
        
        # ONNX inference
        onnx_output = ort_session.run(
            ['logits'],
            {'pixel_values': pixel_values}
        )[0]
        onnx_probs = torch.softmax(torch.tensor(onnx_output), dim=-1)[0]
        
        # Compare outputs
        max_diff = torch.max(torch.abs(pytorch_probs - onnx_probs)).item()
        print(f"✓ ONNX inference successful")
        print(f"  Max difference between PyTorch and ONNX: {max_diff:.6f}")
        
        if max_diff < 1e-5:
            print("  ✓ Outputs match perfectly!")
        elif max_diff < 1e-3:
            print("  ✓ Outputs are very close (acceptable difference)")
        else:
            print("  ⚠ Warning: Outputs differ significantly")
        
        # Display predictions
        print("\nPrediction Comparison:")
        print("-" * 60)
        for i, (pytorch_prob, onnx_prob) in enumerate(zip(pytorch_probs, onnx_probs)):
            label = model.config.id2label[i]
            print(f"{label:10} | PyTorch: {pytorch_prob:.4f} | ONNX: {onnx_prob:.4f}")
    else:
        print(f"⚠ Test image not found at {test_image_path}, skipping comparison")
    
    # Summary
    print("\n" + "=" * 80)
    print("Conversion Complete!")
    print("=" * 80)
    print(f"\nONNX model saved at: {output_path}")
    if os.path.exists(output_path):
        print(f"Model size: {os.path.getsize(output_path) / (1024*1024):.2f} MB")
    
    return output_path

In [ ]:
convert_to_onnx()

In [ ]:
# ============================================================================
# Standalone ONNX Inference (Without Transformers Library)
# ============================================================================
# This code shows how to use the ONNX model for inference without needing
# the transformers library - useful for deployment scenarios
# ============================================================================

import numpy as np
from PIL import Image
import onnxruntime as ort
import torch

def preprocess_image_for_onnx(image_path, size=384):
    """
    Preprocess image for ONNX model inference.
    Replicates the transformers processor behavior.
    
    Args:
        image_path: Path to the image file
        size: Target size (default: 384 for this model)
    
    Returns:
        numpy array ready for ONNX inference
    """
    # Load and resize image
    img = Image.open(image_path).convert('RGB')
    img = img.resize((size, size), Image.BILINEAR)
    
    # Convert to numpy array and rescale to [0, 1]
    img_array = np.array(img).astype(np.float32) / 255.0
    
    # Normalize with mean and std (specific to this model)
    mean = np.array([0.5, 0.5, 0.5], dtype=np.float32)
    std = np.array([0.5, 0.5, 0.5], dtype=np.float32)
    img_array = (img_array - mean) / std
    
    # Convert from HWC (Height, Width, Channels) to CHW (Channels, Height, Width)
    img_array = np.transpose(img_array, (2, 0, 1))
    
    # Add batch dimension: (C, H, W) -> (1, C, H, W)
    img_array = np.expand_dims(img_array, axis=0).astype(np.float32)
    
    return img_array


# Load ONNX model
print("Loading ONNX model...")
onnx_model_path = "./nsfw-image-detection-384.onnx"
session = ort.InferenceSession(onnx_model_path)

# Get model info
print(f"✓ Model loaded: {onnx_model_path}")
print(f"  Input name: {session.get_inputs()[0].name}")
print(f"  Input shape: {session.get_inputs()[0].shape}")
print(f"  Output name: {session.get_outputs()[0].name}")
print(f"  Output shape: {session.get_outputs()[0].shape}")

# Test on multiple images
print("\n" + "=" * 80)
print("Running Standalone ONNX Inference")
print("=" * 80)

import os
image_files = [f for f in os.listdir('./test_images') if f.lower().endswith(('.jpg', '.jpeg', '.png'))]

print(f"\n{'Image':<30} | {'Prediction':<10} | {'Confidence':>10} | Details")
print("-" * 90)

for img_file in image_files:
    img_path = os.path.join('./test_images', img_file)
    
    # Preprocess image
    pixel_values = preprocess_image_for_onnx(img_path, size=384)
    
    # Run ONNX inference
    outputs = session.run(["logits"], {"pixel_values": pixel_values})
    logits = outputs[0]
    
    # Convert to probabilities
    probs = torch.softmax(torch.tensor(logits), dim=-1)[0]
    
    # Get prediction
    pred_id = int(probs.argmax())
    labels = {0: "NSFW", 1: "SFW"}
    prediction = labels[pred_id]
    confidence = probs[pred_id].item()
    
    # Format details
    details = f"NSFW: {probs[0]:.2%} | SFW: {probs[1]:.2%}"
    
    print(f"{img_file:<30} | {prediction:<10} | {confidence:>9.2%} | {details}")

print("\n" + "=" * 80)
print("✓ Standalone ONNX inference completed successfully!")
print("=" * 80)


# TFLITE conversion

In [ ]:
"""
PyTorch to TFLite Conversion Script for NSFW Detection Model
This script converts the Hugging Face PyTorch model to TFLite format with verification.
"""

import os
import torch
import numpy as np
import ai_edge_torch
from transformers import AutoImageProcessor, AutoModelForImageClassification
from PIL import Image


def convert_to_tflite(
    model_path: str = "./nsfw-image-detection-384",
    output_path: str = "./nsfw-image-detection-384.tflite",
    test_image_path: str = "./test_images/normal_1.jpg"
):
    """
    Convert a Hugging Face PyTorch model to TFLite format.
    
    Args:
        model_path: Path to the saved PyTorch model or model name from Hugging Face
        output_path: Path where TFLite model will be saved
        test_image_path: Path to test image for verification
    """
    
    print("=" * 80)
    print("PyTorch to TFLite Conversion")
    print("=" * 80)
    
    # Step 1: Load the model and processor
    print("\n[1/5] Loading PyTorch model and processor...")
    processor = AutoImageProcessor.from_pretrained(model_path, use_fast=False)
    model = AutoModelForImageClassification.from_pretrained(model_path)
    model.eval()
    print("✓ Model loaded successfully")
    
    # Step 2: Create sample input for conversion
    print("\n[2/5] Creating sample input tensor...")
    input_size = 384
    batch_size = 1
    channels = 3
    
    sample_inputs = (torch.randn(batch_size, channels, input_size, input_size),)
    print(f"✓ Sample input shape: {sample_inputs[0].shape}")
    
    # Step 3: Get PyTorch baseline output
    print("\n[3/5] Getting PyTorch baseline output...")
    with torch.no_grad():
        pytorch_output = model(*sample_inputs)
        pytorch_logits = pytorch_output.logits.detach().cpu().numpy()
    print("✓ PyTorch inference complete")
    
    # Step 4: Convert to TFLite
    print("\n[4/5] Converting model to TFLite format...")
    try:
        edge_model = ai_edge_torch.convert(model, sample_inputs)
        print("✓ Model converted to TFLite successfully")
    except Exception as e:
        print(f"✗ Conversion failed: {e}")
        raise
    
    # Step 5: Verify TFLite model
    print("\n[5/5] Verifying TFLite model...")
    
    # Test with sample input
    edge_output = edge_model(*sample_inputs)
    edge_logits = edge_output['logits']
    
    # Compare outputs
    max_diff = np.max(np.abs(pytorch_logits - edge_logits))
    print(f"✓ TFLite inference successful")
    print(f"  Max difference between PyTorch and TFLite: {max_diff:.6f}")
    
    if np.allclose(pytorch_logits, edge_logits, atol=1e-5):
        print("  ✓ Inference result with PyTorch and TFLite was within tolerance")
    elif np.allclose(pytorch_logits, edge_logits, atol=1e-3):
        print("  ✓ Outputs are very close (acceptable difference)")
    else:
        print("  ⚠ Warning: Outputs differ significantly")
        print("  ✗ Something wrong with PyTorch --> TFLite")
    
    # Test with actual image if available
    if os.path.exists(test_image_path):
        print("\nTesting with real image...")
        test_image = Image.open(test_image_path).convert("RGB")
        
        # Prepare input
        inputs = processor(images=test_image, return_tensors="pt")
        pixel_values = inputs['pixel_values']
        
        # PyTorch inference
        with torch.no_grad():
            pytorch_output = model(**inputs).logits
            pytorch_probs = torch.softmax(pytorch_output, dim=-1)[0].numpy()
        
        # TFLite inference
        edge_output = edge_model(pixel_values)
        edge_logits = edge_output['logits']
        edge_probs = torch.softmax(torch.tensor(edge_logits), dim=-1)[0].numpy()
        
        # Compare
        max_diff = np.max(np.abs(pytorch_probs - edge_probs))
        print(f"  Max difference on real image: {max_diff:.6f}")
        
        # Display predictions
        print("\nPrediction Comparison:")
        print("-" * 60)
        for i in range(len(pytorch_probs)):
            label = model.config.id2label[i]
            print(f"{label:10} | PyTorch: {pytorch_probs[i]:.4f} | TFLite: {edge_probs[i]:.4f}")
    else:
        print(f"\n⚠ Test image not found at {test_image_path}, skipping real image comparison")
    
    # Export TFLite model
    print(f"\nExporting TFLite model to: {output_path}")
    edge_model.export(output_path)
    print("✓ Model exported successfully")
    
    # Summary
    print("\n" + "=" * 80)
    print("Conversion Complete!")
    print("=" * 80)
    print(f"\nTFLite model saved at: {output_path}")
    if os.path.exists(output_path):
        print(f"Model size: {os.path.getsize(output_path) / (1024*1024):.2f} MB")
    
    return output_path

In [ ]:
convert_to_tflite(
        model_path="Marqo/nsfw-image-detection-384",
        output_path="./nsfw_detector.tflite",
        test_image_path="./test_images/normal_1.jpg"
    )

In [ ]:
def standalone_tflite_inference(
    model_path: str = "./nsfw_detector.tflite",
    test_images_dir: str = "./test_images",
    size: int = 384
):
    """
    Standalone TFLite inference without transformers library.
    Useful for deployment scenarios.
    
    Args:
        model_path: Path to the TFLite model
        test_images_dir: Directory containing test images
        size: Input size for the model (default: 384)
    """
    # ============================================================================
    # Standalone TFLite Inference (Without Transformers Library)
    # ============================================================================
    # This code shows how to use the TFLite model for inference without needing
    # the transformers library - useful for deployment scenarios
    # ============================================================================
    
    def preprocess_image_for_tflite(image_path, size=384):
        """
        Preprocess image for TFLite model inference.
        Replicates the transformers processor behavior.
        
        Args:
            image_path: Path to the image file
            size: Target size (default: 384 for this model)
        
        Returns:
            numpy array ready for TFLite inference
        """
        # Load and resize image
        img = Image.open(image_path).convert('RGB')
        img = img.resize((size, size), Image.BILINEAR)
        
        # Convert to numpy array and rescale to [0, 1]
        img_array = np.array(img).astype(np.float32) / 255.0
        
        # Normalize with mean and std (specific to this model)
        mean = np.array([0.5, 0.5, 0.5], dtype=np.float32)
        std = np.array([0.5, 0.5, 0.5], dtype=np.float32)
        img_array = (img_array - mean) / std
        
        # Convert from HWC (Height, Width, Channels) to CHW (Channels, Height, Width)
        img_array = np.transpose(img_array, (2, 0, 1))
        
        # Add batch dimension: (C, H, W) -> (1, C, H, W)
        img_array = np.expand_dims(img_array, axis=0).astype(np.float32)
        
        return img_array
    
    # Load model (try ai_edge_litert first, fallback to tensorflow)
    print("Loading TFLite model...")
    try:
        from ai_edge_litert.interpreter import Interpreter
        print("Using ai_edge_litert interpreter")
    except ImportError:
        try:
            from tensorflow.lite import Interpreter
            print("Using TensorFlow Lite interpreter")
        except ImportError:
            print("✗ Error: Neither ai_edge_litert nor tensorflow.lite available")
            print("  Install one of: pip install ai-edge-litert OR pip install tensorflow")
            return
    
    interpreter = Interpreter(model_path=model_path)
    interpreter.allocate_tensors()
    
    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()
    
    # Get model info
    print(f"✓ Model loaded: {model_path}")
    print(f"  Input name: {input_details[0]['name']}")
    print(f"  Input shape: {input_details[0]['shape']}")
    print(f"  Output name: {output_details[0]['name']}")
    print(f"  Output shape: {output_details[0]['shape']}")
    
    # Test on multiple images
    print("\n" + "=" * 80)
    print("Running Standalone TFLite Inference")
    print("=" * 80)
    
    if not os.path.exists(test_images_dir):
        print(f"✗ Test images directory not found: {test_images_dir}")
        return
    
    image_files = [f for f in os.listdir(test_images_dir) 
                   if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    
    if not image_files:
        print(f"✗ No image files found in {test_images_dir}")
        return
    
    print(f"\n{'Image':<30} | {'Prediction':<10} | {'Confidence':>10} | Details")
    print("-" * 90)
    
    for img_file in image_files:
        img_path = os.path.join(test_images_dir, img_file)
        
        # Preprocess image
        pixel_values = preprocess_image_for_tflite(img_path, size=size)
        
        # Run TFLite inference
        interpreter.set_tensor(input_details[0]['index'], pixel_values)
        interpreter.invoke()
        output = interpreter.get_tensor(output_details[0]['index'])
        
        # Get logits
        logits = output[0]
        
        # Convert to probabilities (softmax)
        exp_logits = np.exp(logits - np.max(logits))
        probs = exp_logits / np.sum(exp_logits)
        
        # Get prediction
        pred_id = int(np.argmax(probs))
        labels = {0: "NSFW", 1: "SFW"}
        prediction = labels[pred_id]
        confidence = probs[pred_id]
        
        # Format details
        details = f"NSFW: {probs[0]:.2%} | SFW: {probs[1]:.2%}"
        
        print(f"{img_file:<30} | {prediction:<10} | {confidence:>9.2%} | {details}")
    
    print("\n" + "=" * 80)
    print("✓ Standalone TFLite inference completed successfully!")
    print("=" * 80)


In [ ]:
standalone_tflite_inference(
        model_path="./nsfw_detector.tflite",
        test_images_dir="./test_images",
        size=384
    )